In [1]:
!pip install pycountry

In [2]:
import wbdata
import pandas as pd
import datetime
import os
import pycountry

Key '259617432308479851' not in persistent cache.
Key '7008619839444988308' not in persistent cache.
Key '-8815781626026543860' not in persistent cache.
Key '3481660065358015752' not in persistent cache.
Key '6746492167356574058' not in persistent cache.
Key '-9150647835816715328' not in persistent cache.
Key '5151976267457794123' not in persistent cache.
Key '-8588086479394991788' not in persistent cache.
Key '-6237441455242851459' not in persistent cache.
Key '783258535979536117' not in persistent cache.
Key '8682342844379727601' not in persistent cache.
Key '-2101211744346897581' not in persistent cache.
Key '-3949965056795003479' not in persistent cache.
Key '-3695727644245677516' not in persistent cache.
Key '7521162885019713500' not in persistent cache.
Key '-4176578781553329659' not in persistent cache.
Key '2844121209669057917' not in persistent cache.
Key '-7703538555640302215' not in persistent cache.
Key '-4438706453641743909' not in persistent cache.
Key '-18891023831054816

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")

In [4]:
# 1. Define the indicators you want to fetch
# indicators = {
#     "SP.POP.GROW": "pop_growth",
#     "SP.POP.TOTL": "pop_total",
#     "SP.URB.TOTL.IN.ZS": "pop_urban_pct",
#     "SI.POV.DDAY": "poverty_headcount_ratio",
#     "EG.ELC.ACCS.ZS": "access_to_electricity_pct",
#     "EG.USE.ELEC.KH.PC": "electricity_consumption_per_capita_kwh",
#     "EG.USE.PCAP.KG.OE": "energy_use_per_capita_kg_of_oil_equivalent",
#     "EG.USE.COMM.GD.PP.KD": "energy_use_kg_of_oil_equivalent_per_gdp",
#     "EG.ELC.COAL.ZS": "electricity_from_coal_pct",
#     "EG.ELC.NGAS.ZS": "electricity_from_natural_gas_pct",
#     "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
#     "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
#     "AG.LND.FRST.ZS": "forest_area_pct",
#     "NV.AGR.TOTL.ZS": "agriculture_value_added_pct",
#     "ER.H2O.FWTL.ZS": "annual_freshwater_withdrawals_pct"
# }


# indicators = {
#     "SP.POP.TOTL": "pop_total",
#     "SP.POP.GROW": "pop_growth",
#     "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
#     "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
#     "AG.LND.FRST.ZS": "forest_area_pct",
#     "NY.GDP.MKTP.KD": "gdp_2015_usd",
#     "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
#     "NY.GDP.PCAP.KD": "gdp_per_capita_2015_usd",
#     "NY.GDP.PCAP.KD.ZG": "gdp_per_capita_growth_pct",
#     "NE.IMP.GNFS.KD": "imports_2015_usd",
#     "NE.IMP.GNFS.ZS": "imports_pct_of_gdp",
#     "NV.IND.TOTL.KD": "industry_value_added_2015_usd",
#     "NV.IND.TOTL.ZS": "industry_pct_of_gdp",
#     "NV.IND.MANF.KD": "manufacturing_value_added_2015_usd",
#     "NV.IND.MANF.ZS": "manufacturing_pct_of_gdp",
#     "NE.EXP.GNFS.KD": "exports_2015_usd",
#     "NE.EXP.GNFS.ZS": "exports_pct_of_gdp",

# }


indicators = {
    "SP.POP.TOTL": "pop_total",
    "SP.POP.GROW": "pop_growth",
    "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
    "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
    "AG.LND.FRST.ZS": "forest_area_pct",
    "NY.GDP.MKTP.PP.KD": "gdp_2021_ppp_intl_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
    "NY.GDP.PCAP.PP.KD": "gdp_per_capita_2021_ppp_intl_usd",
    "NY.GDP.PCAP.KD.ZG": "gdp_per_capita_growth_pct",
    "NE.IMP.GNFS.ZS": "imports_pct_of_gdp",
    "NV.IND.TOTL.ZS": "industry_pct_of_gdp",
    "NV.IND.MANF.ZS": "manufacturing_pct_of_gdp",
    "NE.EXP.GNFS.ZS": "exports_pct_of_gdp",

}

In [5]:
date_range = (datetime.datetime(2000,1,1), datetime.datetime(2022,1,1))

df = wbdata.get_dataframe(
    indicators,
    country="all",
    date=date_range,
    freq='Y',
    parse_dates=True
).reset_index().rename(columns={'country':'region','date':'year'})

# 2) Pull the country lookup (wbdata.get_countries returns a list of dicts)
country_list = wbdata.get_countries()  

#  Inspect one entry to see its keys:
# print(country_list[0])
# -> {'id': 'ABW', 'iso2Code': 'AW', 'name': 'Aruba', …}

# 3) Build a small DataFrame of name → ISO-2
country_df = pd.DataFrame(country_list)[['name','iso2Code']]
country_df.columns = ['region','iso2']  

# 4) Merge it onto your indicators DataFrame
df = df.merge(country_df, on='region', how='left')

# 5) (Optional) Convert ISO-2 → ISO-3 with pycountry
def iso2_to_iso3(c2):
    try:
        return pycountry.countries.get(alpha_2=c2).alpha_3
    except:
        return None

df['iso_alpha_3'] = df['iso2'].map(iso2_to_iso3)

# make year only the year
df['year'] = df['year'].dt.year

# drop iso2 and region
df = df.drop(columns=['iso2', 'region'])


Key '8473363730313318183' not in persistent cache.


In [6]:
df

,year,pop_total,pop_growth,electricity_from_renewables_pct,renewable_energy_consumption_pct,forest_area_pct,gdp_2021_ppp_intl_usd,gdp_growth_pct,gdp_per_capita_2021_ppp_intl_usd,gdp_per_capita_growth_pct,imports_pct_of_gdp,industry_pct_of_gdp,manufacturing_pct_of_gdp,exports_pct_of_gdp,iso_alpha_3
0,2022,731821393.0,2.592754,NaN,NaN,29.737205,2.908437e+12,3.555769,3974.244214,0.905330,30.011775,26.932515,10.538783,27.461822,None
1,2021,713090928.0,2.649439,4.939318,NaN,29.955194,2.805001e+12,4.563568,3933.580905,1.829597,26.344785,26.085775,10.411407,25.885464,None
2,2020,694446100.0,2.699516,4.585792,65.78238,30.174252,2.681304e+12,-2.859784,3861.068816,-5.447021,24.256267,25.435730,10.447498,22.041722,None
3,2019,675950189.0,2.721681,5.173796,62.69071,30.391626,2.753587e+12,2.200340,4073.653989,-0.543715,26.740975,26.422837,10.784120,23.926080,None
4,2018,657801085.0,2.734263,3.945917,61.58753,30.611512,2.689356e+12,2.665038,4088.402831,-0.104064,28.520850,27.858301,10.839451,25.306400,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,2004,12365896.0,1.086049,0.893713,81.80000,46.999354,3.498118e+10,-5.807538,2828.843352,-6.824979,41.569810,24.382749,13.949485,34.469799,ZWE
6114,2003,12232323.0,1.189736,0.595068,78.00000,47.118444,3.713799e+10,-16.995075,3036.053357,-17.976763,38.054929,NaN,12.166740,32.397059,ZWE
6115,2002,12087653.0,0.962220,0.815768,74.70000,47.237534,4.474191e+10,-8.894024,3701.455163,-9.766459,34.972553,NaN,11.871740,31.834799,ZWE
6116,2001,11971901.0,0.669179,1.686061,72.30000,47.356624,4.910974e+10,1.439615,4102.083474,0.763069,32.938959,NaN,13.145372,34.958913,ZWE


In [7]:
df.to_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_indicators.csv'), index=False)